In [2]:
import duckdb
import pandas as pd

# Connect to a local database file
con = duckdb.connect("../data/processed/provider_quality.db")

# Load raw CSVs into DuckDB tables
con.execute("""
    CREATE OR REPLACE TABLE hospitals AS
    SELECT * FROM read_csv_auto('../data/raw/hospitals.csv')
""")

con.execute("""
    CREATE OR REPLACE TABLE complications AS
    SELECT * FROM read_csv_auto('../data/raw/complications.csv')
""")

# Quick check
print("hospitals:", con.execute("SELECT COUNT(*) FROM hospitals").fetchone()[0])
print("complications:", con.execute("SELECT COUNT(*) FROM complications").fetchone()[0])

hospitals: 5432
complications: 95840


In [3]:
# What does the hospitals table look like?
con.execute("SELECT * FROM hospitals LIMIT 3").df()

,Facility ID,Facility Name,Address,City/Town,State,ZIP Code,County/Parish,Telephone Number,Hospital Type,Hospital Ownership,...,Count of READM Measures Better,Count of READM Measures No Different,Count of READM Measures Worse,READM Group Footnote,Pt Exp Group Measure Count,Count of Facility Pt Exp Measures,Pt Exp Group Footnote,TE Group Measure Count,Count of Facility TE Measures,TE Group Footnote
0,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,Acute Care Hospitals,Government - Hospital District or Authority,...,1,9,1,<NA>,15,15,<NA>,10,10,<NA>
1,010005,MARSHALL MEDICAL CENTERS,2505 U S HIGHWAY 431 NORTH,BOAZ,AL,35957,MARSHALL,(256) 593-8310,Acute Care Hospitals,Government - Hospital District or Authority,...,1,8,0,<NA>,15,15,<NA>,10,10,<NA>
2,010006,NORTH ALABAMA MEDICAL CENTER,1701 VETERANS DRIVE,FLORENCE,AL,35630,LAUDERDALE,(256) 768-8400,Acute Care Hospitals,Proprietary,...,1,8,0,<NA>,15,15,<NA>,10,9,<NA>


In [4]:
# What does the complications table look like?
con.execute("SELECT * FROM complications LIMIT 3").df()

,Facility ID,Facility Name,Address,City/Town,State,ZIP Code,County/Parish,Telephone Number,Measure ID,Measure Name,Compared to National,Denominator,Score,Lower Estimate,Higher Estimate,Footnote,Start Date,End Date
0,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,COMP_HIP_KNEE,Rate of complications for hip/knee replacement...,No Different Than the National Rate,27,3.2,1.7,5.9,None,2021-01-04,03/31/2024
1,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,Hybrid_HWM,Hybrid Hospital-Wide All-Cause Risk Standardiz...,No Different Than the National Rate,1835,4.5,2.6,7.4,None,2023-01-07,06/30/2024
2,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,MORT_30_AMI,Death rate for heart attack patients,No Different Than the National Rate,270,11.4,9.1,14.3,None,2021-01-07,06/30/2024


In [5]:
# What quality measures are available?
con.execute("""
    SELECT 
        "Measure ID",
        "Measure Name",
        COUNT(DISTINCT "Facility ID") as num_hospitals
    FROM complications
    GROUP BY "Measure ID", "Measure Name"
    ORDER BY num_hospitals DESC
""").df()

,Measure ID,Measure Name,num_hospitals
0,MORT_30_COPD,Death rate for COPD patients,4792
1,PSI_09,Postoperative hemorrhage or hematoma rate,4792
2,PSI_06,Iatrogenic pneumothorax rate,4792
3,PSI_04,Death rate among surgical inpatients with seri...,4792
4,PSI_15,Abdominopelvic accidental puncture or lacerati...,4792
5,PSI_13,Postoperative sepsis rate,4792
6,PSI_90,CMS Medicare PSI 90: Patient safety and advers...,4792
7,Hybrid_HWM,Hybrid Hospital-Wide All-Cause Risk Standardiz...,4792
8,MORT_30_STK,Death rate for stroke patients,4792
9,MORT_30_AMI,Death rate for heart attack patients,4792


In [6]:
# How many hospitals perform worse than national average for heart attack mortality?
con.execute("""
    SELECT 
        "Compared to National",
        COUNT(*) as num_hospitals,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) as pct
    FROM complications
    WHERE "Measure ID" = 'MORT_30_AMI'
      AND "Compared to National" IS NOT NULL
    GROUP BY "Compared to National"
    ORDER BY num_hospitals DESC
""").df()

,Compared to National,num_hospitals,pct
0,No Different Than the National Rate,1910,39.9
1,Number of Cases Too Small,1796,37.5
2,Not Available,1044,21.8
3,Better Than the National Rate,29,0.6
4,Worse Than the National Rate,13,0.3


In [7]:
# Which hospitals are worse than national average for heart attack mortality?
con.execute("""
    SELECT 
        c."Facility ID",
        c."Facility Name",
        h."State",
        h."Hospital Type",
        c."Score",
        c."Denominator"
    FROM complications c
    JOIN hospitals h 
        ON c."Facility ID" = h."Facility ID"
    WHERE c."Measure ID" = 'MORT_30_AMI'
      AND c."Compared to National" = 'Worse Than the National Rate'
    ORDER BY CAST(c."Score" AS FLOAT) DESC
""").df()

,Facility ID,Facility Name,State,Hospital Type,Score,Denominator
0,010039,HUNTSVILLE HOSPITAL,AL,Acute Care Hospitals,17.1,541
1,450133,MIDLAND MEMORIAL HOSPITAL,TX,Acute Care Hospitals,16.7,131
2,250082,DELTA HEALTH SYSTEM - THE MEDICAL CENTER,MS,Acute Care Hospitals,16.5,116
3,340143,CATAWBA VALLEY MEDICAL CENTER,NC,Acute Care Hospitals,16.2,114
4,440228,SAINT FRANCIS BARTLETT MEDICAL CENTER,TN,Acute Care Hospitals,16.1,148
5,050107,MARIAN REGIONAL MEDICAL CENTER,CA,Acute Care Hospitals,15.8,243
6,170020,HUTCHINSON REGIONAL MEDICAL CENTER INC,KS,Acute Care Hospitals,15.8,181
7,100157,LAKELAND REGIONAL MEDICAL CENTER,FL,Acute Care Hospitals,15.7,316
8,340070,ALAMANCE REGIONAL MEDICAL CENTER,NC,Acute Care Hospitals,15.7,125
9,440035,TENNOVA HEALTHCARE - CLARKSVILLE,TN,Acute Care Hospitals,15.5,184


## Finding 1: Hospitals Worse Than National Average for Heart Attack Mortality

Of 4,792 hospitals with data on 30-day heart attack mortality (`MORT_30_AMI`), 
only **13 hospitals (0.3%)** perform statistically worse than the national rate.

The `Score` column represents the 30-day mortality rate — the percentage of 
patients who die within 30 days of a heart attack admission. The national 
average is approximately 13%. These 13 hospitals range from 14.4% to 17.1%, 
meaning up to **4 additional deaths per 100 patients** compared to the national benchmark.

Note: 37.5% of hospitals had too few cases to evaluate, and 21.8% had no data 
available — a critical consideration for any provider recommendation model.

**Key SQL concepts used:** `JOIN`, `WHERE` filter, `CAST`, window function (`OVER()`).